<div style="background: linear-gradient(135deg, #1e293b 0%, #0f172a 100%); padding: 30px; border-radius: 8px; margin-bottom: 25px; box-shadow: 0 4px 6px -1px rgba(0, 0, 0, 0.1);">
    <h1 style="color: #f8fafc; font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif; font-weight: 700; margin: 0; font-size: 2.2em; letter-spacing: -0.5px;"> Procesamiento datos deberes PISA 🌍
</h1>
    <p style="color: #94a3b8; font-family: 'Segoe UI', sans-serif; margin: 8px 0 0 0; font-size: 1.1em;">Obtención de los datos para el analisis de al eficiencia del estudio (deberes) </p>
    
</div>

# 📋 Descripción  

Este cuaderno registra la obtención de datos para el estudio de la eficiencia del estudio de los alumnos, partiremos de la submuestra de colegios generada en el análisis global incorporando los datos necesarior para este apartado.

Seleccion de variables:

**Estudiantes (`STU_QQQ_SAS`)**

| Código de variable | Descripción y rol en el proyecto |
| :--- | :--- |
| `ST296Q01JA` | Tiempo diario dedicado a los deberes de Matemáticas. |
| `ST296Q04JA` | Tiempo total acumulado diario en deberes de todas las asignaturas. |
| `STUDYHMW` | Índice derivado de frecuencia semanal de estudio antes/después de clase. |
| `ST307Q07JA` | Perseverancia: *"Dejo de hacer los deberes si son demasiado largos"*. |
| `ST309Q05JA` | Autorregulación: *"Reviso cuidadosamente los deberes antes de entregarlos"*. |
| `ESCS` | Índice socioeconómico individual (control obligatorio). |
| `PERSEVAGR` / `MATHPERS` | Índices estandarizados de perseverancia general y esfuerzo en matemáticas. |
| `W_FSTUWT` / `SENWT` / `STRATUM`  (Ajustados)| Pesos de muestreo (final, senado y estrato) para el diseño muestral. |

 **Escuelas (`SCH_QQQ_SAS`)**

| Código de variable | Descripción y rol en el proyecto |
| :--- | :--- |
| `SC212Q01JA` | Disponibilidad de salas donde el alumnado puede hacer los deberes. |
| `SC212Q02JA` | Disponibilidad de personal del centro que ayuda con los deberes. |
| `SC212Q03JA` | Disponibilidad de tutoría de deberes entre iguales (*peer-to-peer*). |

**Tiempos (`STU_TIM_SAS`)**

| Código de variable | Descripción y rol en el proyecto |
| :--- | :--- |
| `ST296_TT`, `ST307_TT`, `ST309_TT` | Tiempo de respuesta asociado a los bloques de preguntas sobre deberes, perseverancia y autorregulación. |

**Cognitivo (`STU_COG_SAS`)**

| Código de variable | Descripción y rol en el proyecto |
| :--- | :--- |
| `PV1MATH` a `PV10MATH` | Los 10 Valores Plausibles de matemáticas (**variables objetivo / target**). |

# Preparación de los datos

Se carga la submuestra de escuelas creada en el cuaderno global y se prepara la lectura por bloques del fichero de estudiantes. La tabla inicial contendrá las variables de deberes y los identificadores necesarios para filtrar y relacionar los datos.

In [6]:
import gc
import os
import pickle
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import pyreadstat
from src.utils_pisa import *


In [4]:
#Se indica la ruta del workspace principal (Dos niveles atras)
sys.path.append(os.path.abspath('../../'))

#Se indica la comprobacion de cambio de librerias externas
%load_ext autoreload
%autoreload 2

In [7]:

ruta_datos_raw = Path('../../data/raw')
ruta_datos_intermedios = Path('../../data/intermediate')
ruta_datos_estudiante = ruta_datos_raw / 'CY08MSP_STU_QQQ.SAS7BDAT'
ruta_datos_escuelas = ruta_datos_raw / 'CY08MSP_SCH_QQQ.SAS7BDAT'
ruta_datos_tiempos = ruta_datos_raw / 'CY08MSP_STU_TIM.SAS7BDAT'
ruta_submuestra = ruta_datos_intermedios / 'escuelas_seleccionadas.pkl'


In [8]:
establecer_semilla()

✅ Semillas del proyecto establecidas con éxito (seed=42).


## Lectura de estudiantes por bloques

Se conservan las variables de interés del cuestionario de estudiantes y las claves que permiten filtrar las escuelas seleccionadas.

In [ ]:
#Cargamos el set de datos donde se indica el ID de las escuelas seleccionadas y los factoresd e inflacion
escuelas_seleccionadas_set, factores_dict = cargar_pickle(ruta_submuestra)

#Cargamos los pesos de estudiante ajustados de la submuestra

#Cargamos los pesos de senado ajustados de la submuestra



 Objetos cargados correctamente desde: ..\..\data\intermediate\escuelas_seleccionadas.pkl


In [19]:
list(escuelas_seleccionadas_set)[:5]

[('KHM', 'KHM05', np.float64(11600098.0)),
 ('LTU', 'LTU09', np.float64(44000270.0)),
 ('JPN', 'JPN01', np.float64(39200149.0)),
 ('AUS', 'AUS05', np.float64(3600495.0)),
 ('QUR', 'UKR25', np.float64(90100156.0))]

In [ ]:
#Establecemos las columnas de interes para cada uno de los dataframes que vamos a procesar

#STU_QQQ.SAS7BDAT
columnas_estudiantes = [
    'CNT', 'STRATUM', 'CNTSCHID', 'CNTSTUID',
    'ST296Q01JA', 'ST296Q04JA', 'STUDYHMW',
    'ST307Q07JA', 'ST309Q05JA', 'ESCS',
    'PERSEVAGR', 'MATHPERS'
]

#SCH_QQQ.SAS7BDAT
columnas_escuelas = [
    'CNT', 'STRATUM', 'CNTSCHID',
    'SC212Q01JA', 'SC212Q02JA', 'SC212Q03JA'
]

#STU_TIM.SAS7BDAT
columnas_tiempos = [
    'CNT', 'CNTSTUID', 'ST296_TT', 'ST307_TT', 'ST309_TT'
]


In [ ]:
#Reader para el archivo de estudiantes en chunks (el mas grande de todos) 
chunk_size = 10_000
reader = pyreadstat.read_file_in_chunks(
    pyreadstat.read_sas7bdat,
    str(ruta_datos_estudiante),
    chunksize=chunk_size,
    usecols=columnas_estudiantes
)

In [20]:

chunks_filtrados = []

for chunk, _ in reader:
    #Creamos la clave temporal en forma de tupla en cada chunk
    chunk['clave_colegio_temp'] = list(zip(
        chunk['CNT'],
        chunk['STRATUM'],
        chunk['CNTSCHID']
    ))

    #De cada chunk nos quedamos solo con las escuelas presentes en el set
    chunk_filtrado = chunk.loc[
        chunk['clave_colegio_temp'].isin(escuelas_seleccionadas_set),
        columnas_estudiantes
    ].copy()

    if not chunk_filtrado.empty:
        chunks_filtrados.append(chunk_filtrado) #Añadimos el chunk filtrado resultante a la lista de chunks

if not chunks_filtrados:
    raise ValueError('La lectura no encontró estudiantes de las escuelas seleccionadas.')

df_estudiantes_submuestra = pd.concat(chunks_filtrados, ignore_index=True)
del chunks_filtrados
gc.collect()

df_estudiantes_submuestra.shape

(146500, 14)

In [23]:
list(df_estudiantes_submuestra.columns)

['CNT',
 'STRATUM',
 'CNTSCHID',
 'CNTSTUID',
 'W_FSTUWT',
 'SENWT',
 'ST296Q01JA',
 'ST296Q04JA',
 'STUDYHMW',
 'ST307Q07JA',
 'ST309Q05JA',
 'ESCS',
 'PERSEVAGR',
 'MATHPERS']

## Lectura de escuelas & Tiempos

Se cargan las variables de apoyo escolar disponibles en el fichero de escuelas y los tiempos de respuesta de los bloques relacionados con deberes. En ambos casos se filtran los registros de la submuestra.

In [ ]:
# El fichero de escuelas es reducido; se carga solo con las columnas necesarias.
df_escuelas = pyreadstat.read_sas7bdat(
    str(ruta_datos_escuelas),
    usecols=columnas_escuelas
)[0]

df_escuelas['clave_colegio_temp'] = list(zip(
    df_escuelas['CNT'],
    df_escuelas['STRATUM'],
    df_escuelas['CNTSCHID']
))
df_escuelas_submuestra = df_escuelas.loc[
    df_escuelas['clave_colegio_temp'].isin(escuelas_seleccionadas_set),
    columnas_escuelas
].copy()
del df_escuelas

if df_escuelas_submuestra.empty:
    raise ValueError('La lectura no encontró escuelas de la submuestra en el fichero de escuelas.')

df_escuelas_submuestra.shape

In [ ]:
# Los tiempos se registran por estudiante, por eso se leen en bloques.
claves_estudiantes = set(zip(
    df_estudiantes_submuestra['CNT'],
    df_estudiantes_submuestra['CNTSTUID']
))

reader_tiempos = pyreadstat.read_file_in_chunks(
    pyreadstat.read_sas7bdat,
    str(ruta_datos_tiempos),
    chunksize=chunk_size,
    usecols=columnas_tiempos
)

chunks_tiempos_filtrados = []
for chunk_tiempos, _ in reader_tiempos:
    claves_chunk = list(zip(chunk_tiempos['CNT'], chunk_tiempos['CNTSTUID']))
    chunk_tiempos_filtrado = chunk_tiempos.loc[
        pd.Series(claves_chunk, index=chunk_tiempos.index).isin(claves_estudiantes),
        columnas_tiempos
    ].copy()

    if not chunk_tiempos_filtrado.empty:
        chunks_tiempos_filtrados.append(chunk_tiempos_filtrado)

if not chunks_tiempos_filtrados:
    raise ValueError('La lectura no encontró registros de tiempo para los estudiantes de la submuestra.')

df_tiempos_submuestra = pd.concat(chunks_tiempos_filtrados, ignore_index=True)
del chunks_tiempos_filtrados, claves_chunk
gc.collect()

df_tiempos_submuestra.shape

In [ ]:
# Añadimos variables escolares y de tiempo a la tabla de estudiantes.
df_analisis_deberes = df_estudiantes_submuestra.merge(
    df_escuelas_submuestra.drop(columns=['CNT', 'STRATUM', 'CNTSCHID']),
    on=['CNT', 'STRATUM', 'CNTSCHID'],
    how='left',
    validate='many_to_one'
).merge(
    df_tiempos_submuestra,
    on=['CNT', 'CNTSTUID'],
    how='left',
    validate='one_to_one'
)

df_analisis_deberes.shape

Lo que tengo que hacer esl coger los pesos de senado y los de W_FSWT ajustados y ponerlos en el dataframe ( como ya estan calculados en el global solamente habría que hacer un merge)